In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from collections import Counter
import re
import os

In [ ]:
CSV_PATH    = "/content/drive/MyDrive/Sem 8/NLP/Mammo.csv"
TEXT_COLUMN = "Features"
EMBED_DIM   = 64
WINDOW_SIZE = 2
BATCH_SIZE  = 64
EPOCHS      = 100
LR          = 0.001
MIN_FREQ    = 1

In [ ]:
def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

In [ ]:
data = pd.read_csv(CSV_PATH)
data.head()

,Features,Birads
0,Soft tissue mass lesion (23 x 20 mm) with spic...,4
1,Parenchyma is predominantly FATTY.No distinctl...,1
2,Parenchyma is predominantly GLANDULAR. No dist...,1
3,Parenchyma is predominantly GLANDULAR. No dist...,2
4,Parenchyma is GLANDULAR and FATTY. Small subce...,2


In [ ]:
df = data[TEXT_COLUMN].dropna().astype(str)
sentences = [tokenize(s) for s in df]
sentences = [s for s in sentences if len(s) > 1]

In [ ]:
def build_vocab(sentences: list, min_freq: int = 1):
    freq = Counter(w for s in sentences for w in s)
    tokens = [w for w, c in freq.items() if c >= min_freq]

    word2idx = {w: i for i, w in enumerate(tokens)}
    idx2word = {i: w for w, i in word2idx.items()}
    vocab_size = len(tokens)

    return word2idx, idx2word, vocab_size

def encode(sentence: list, word2idx: dict) -> list:
    return [word2idx[w] for w in sentence if w in word2idx]

In [ ]:
word2idx, idx2word, vocab_size = build_vocab(sentences, min_freq=MIN_FREQ)
encoded = [encode(s, word2idx) for s in sentences]

In [ ]:
class CBOWDataset(Dataset):
    def __init__(self, sentences, word2idx: dict, window_size: int = 2):
        self.data = []
        for sentence in sentences:
            encoded = encode(sentence, word2idx)
            for i, target in enumerate(encoded):
                context = [
                    encoded[j]
                    for j in range(i - window_size, i + window_size + 1)
                    if j != i and 0 <= j < len(encoded)
                ]
                if context:
                    self.data.append((context, target))
        print(f"CBOW dataset: {len(self.data)} (context, target) pairs")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)


def cbow_collate(batch):
    """Pad variable-length context lists to the same length."""
    contexts, targets = zip(*batch)
    max_len = max(c.size(0) for c in contexts)
    padded  = torch.zeros(len(contexts), max_len, dtype=torch.long)
    for i, c in enumerate(contexts):
        padded[i, :c.size(0)] = c
    return padded, torch.stack(targets)


class SkipGramDataset(Dataset):
    def __init__(self, sentences, word2idx: dict, window_size: int = 2):
        self.data = []
        for sentence in sentences:
            encoded = encode(sentence, word2idx)
            for i, center in enumerate(encoded):
                for j in range(i - window_size, i + window_size + 1):
                    if j != i and 0 <= j < len(encoded):
                        self.data.append((center, encoded[j]))
        print(f"Skip-Gram dataset: {len(self.data)} (center, context) pairs")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        center, context = self.data[idx]
        return torch.tensor(center, dtype=torch.long), torch.tensor(context, dtype=torch.long)


In [ ]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear     = nn.Linear(embed_dim, vocab_size)

    def forward(self, context_ids):
        # context_ids : (batch, context_len)
        embeds    = self.embeddings(context_ids)   # (batch, ctx_len, embed_dim)
        avg_embed = embeds.mean(dim=1)             # (batch, embed_dim)
        return self.linear(avg_embed)              # (batch, vocab_size)


class SkipGramModel(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear     = nn.Linear(embed_dim, vocab_size)

    def forward(self, center_ids):
        # center_ids : (batch,)
        embeds = self.embeddings(center_ids)   # (batch, embed_dim)
        return self.linear(embeds)

In [ ]:
def train_model(model, dataloader, epochs: int, lr: float, name: str = "Model"):
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*55}")
    print(f"  Training {name}  |  device: {device}")
    print(f"{'='*55}")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  |  Avg Loss: {total_loss / len(dataloader):.4f}")

    return model

In [ ]:
def most_similar(word: str, model, word2idx: dict, idx2word: dict, top_k: int = 5):
    """Find nearest neighbours by cosine similarity in embedding space."""
    if word not in word2idx:
        print(f"  '{word}' not in vocabulary.")
        return

    W   = model.embeddings.weight.detach().cpu().numpy()
    idx = word2idx[word]
    q   = W[idx]

    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-9
    sims  = (W / norms) @ (q / (np.linalg.norm(q) + 1e-9))

    top = np.argsort(sims)[::-1][1: top_k + 1]
    print(f"\n  Most similar to '{word}':")
    for i in top:
        print(f"    {idx2word[i]:<20s}  cos_sim: {sims[i]:.4f}")

In [ ]:
cbow_ds     = CBOWDataset(sentences, word2idx, WINDOW_SIZE)
cbow_loader = DataLoader(cbow_ds, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=cbow_collate)
cbow_model  = CBOWModel(vocab_size, EMBED_DIM)
cbow_model  = train_model(cbow_model, cbow_loader, EPOCHS, LR, name="CBOW")

# 4. Skip-Gram
sg_ds     = SkipGramDataset(sentences, word2idx, WINDOW_SIZE)
sg_loader = DataLoader(sg_ds, batch_size=BATCH_SIZE, shuffle=True)
sg_model  = SkipGramModel(vocab_size, EMBED_DIM)
sg_model  = train_model(sg_model, sg_loader, EPOCHS, LR, name="Skip-Gram")

CBOW dataset: 7041 (context, target) pairs

  Training CBOW  |  device: cpu
  Epoch   1/100  |  Avg Loss: 5.4811
  Epoch   5/100  |  Avg Loss: 2.2560
  Epoch  10/100  |  Avg Loss: 1.3725
  Epoch  15/100  |  Avg Loss: 0.9538
  Epoch  20/100  |  Avg Loss: 0.7242
  Epoch  25/100  |  Avg Loss: 0.5730
  Epoch  30/100  |  Avg Loss: 0.4759
  Epoch  35/100  |  Avg Loss: 0.3861
  Epoch  40/100  |  Avg Loss: 0.3247
  Epoch  45/100  |  Avg Loss: 0.2792
  Epoch  50/100  |  Avg Loss: 0.2420
  Epoch  55/100  |  Avg Loss: 0.2178
  Epoch  60/100  |  Avg Loss: 0.1912
  Epoch  65/100  |  Avg Loss: 0.1704
  Epoch  70/100  |  Avg Loss: 0.1637
  Epoch  75/100  |  Avg Loss: 0.1453
  Epoch  80/100  |  Avg Loss: 0.1317
  Epoch  85/100  |  Avg Loss: 0.1249
  Epoch  90/100  |  Avg Loss: 0.1172
  Epoch  95/100  |  Avg Loss: 0.1115
  Epoch 100/100  |  Avg Loss: 0.1045
Skip-Gram dataset: 27522 (center, context) pairs

  Training Skip-Gram  |  device: cpu
  Epoch   1/100  |  Avg Loss: 4.8467
  Epoch   5/100  |  Avg

In [ ]:
sample_word = list(word2idx.keys())[50]
print(f"\nSample word: '{sample_word}'")
print("\n" + "="*55)
print("  Embedding Similarity")
print("="*55)
print("\n[CBOW]")
most_similar(sample_word, cbow_model, word2idx, idx2word)
print("\n[Skip-Gram]")
most_similar(sample_word, sg_model, word2idx, idx2word)


Sample word: 'or'

  Embedding Similarity

[CBOW]

  Most similar to 'or':
    microcalcificationsvascular  cos_sim: 0.3854
    thickening            cos_sim: 0.3212
    margins               cos_sim: 0.2925
    lymphadenopathynormal  cos_sim: 0.2843
    parenchymal           cos_sim: 0.2814

[Skip-Gram]

  Most similar to 'or':
    thickening            cos_sim: 0.4732
    irregularity          cos_sim: 0.4268
    irregularityright     cos_sim: 0.3495
    are                   cos_sim: 0.2905
    multifocal            cos_sim: 0.2851


In [ ]:
def get_word_embedding(word: str, model, word2idx: dict):
    """Returns the embedding vector for a given word."""
    if word not in word2idx:
        print(f"  '{word}' not in vocabulary.")
        return None

    idx = word2idx[word]
    # Get the embedding vector from the model's embedding layer
    embedding = model.embeddings.weight[idx].detach().cpu().numpy()
    return embedding


word_to_find = 'mass'
embedding_vector = get_word_embedding(word_to_find, cbow_model, word2idx)

if embedding_vector is not None:
    print(f"Embedding for '{word_to_find}' (CBOW model):\n{embedding_vector}")
    print(f"Shape of embedding: {embedding_vector.shape}")

Embedding for 'mass' (CBOW model):
[-1.4759572  -3.7705295   1.1194423  -3.4527063  -0.04738105 -1.1190379
  2.7251985  -3.9210138  -1.8530849  -2.3429403  -1.1064157   0.23645976
  2.1905434  -1.5880218  -0.46261436 -0.5360902   2.934427   -3.6339781
 -3.4205523  -2.019982   -1.0975504  -0.29840088  2.716375   -0.300011
  1.0727805  -0.35451552 -0.44372007  0.8329269   0.39092848  2.4454181
 -0.13756469  1.922562   -1.9055555  -0.20670405  1.3567302  -0.4905978
  2.9807196   0.6351784  -1.221273    1.1091342   1.903028    2.596401
 -1.1989182   1.9455796  -0.43567824  2.5606182  -1.8836536  -0.9652798
  1.6799439  -0.7677572  -0.3154387   0.2945259  -2.5435145  -1.2163533
 -0.5048031  -1.0225456  -0.97321033 -0.9037168  -0.48165348  1.77627
  0.57111704  2.1219077   1.3747686  -0.5651756 ]
Shape of embedding: (64,)
